# Figure S6 — manual clusters (rebuilt from old notebook, plot-faithful)

This notebook reproduces the old S6 manual-cluster trends while applying the required fixes:
- local normalization formula is preserved as $freq / local\_mean$
- x-axis is numeric milliseconds (Kaleido-safe)
- 2014 window is shifted by `+8 years, +6 days`
- saved outputs keep the same style and export behavior

Output folder: `outputs/figures/Fig.S6_manual_clusters/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from config import WORD_FORMS_ALL, CHOSEN_WORDS_DAILY_FILE, FIGURES_DIR

warnings.filterwarnings('ignore')
pio.templates.default = 'plotly_white'

In [ ]:
# --- Window settings and output paths ---
CENTRAL_DATES = {
    '2014': pd.to_datetime('2014-02-18'),
    '2022': pd.to_datetime('2022-02-24')
}

S6_OUT = FIGURES_DIR / 'Fig.S6_manual_clusters'
S6_BASE = S6_OUT
S6_OUT.mkdir(parents=True, exist_ok=True)

lang2cluster = {
    # Cluster 1: persistently high attention
    'Ukrainian': 'Stable high attention',
    'Russian':   'Stable high attention',

    # Cluster 2: short-lived, transcontinental spikes
    'Turkish':    'Short-lived reaction',
    'Portuguese': 'Short-lived reaction',
    'Arabic':     'Short-lived reaction',
    'Catalan':    'Short-lived reaction',
    'Korean':     'Short-lived reaction',
    'Persian':    'Short-lived reaction',

    # Cluster 3: strong 2014 spike only
    'Estonian':   '2014-sustained only',
    'Serbian':    '2014-sustained only',
    'Hungarian':  '2014-sustained only',
    'Vietnamese': '2014-sustained only',
    'Urdu':       '2014-sustained only',

    # Cluster 4: muted in 2014 -> strong, prolonged 2022
    'German':    'EU-heavy 2022 reaction',
    'Italian':   'EU-heavy 2022 reaction',
    'French':    'EU-heavy 2022 reaction',
    'Finnish':   'EU-heavy 2022 reaction',
    'Norwegian': 'EU-heavy 2022 reaction',
    'Dutch':     'EU-heavy 2022 reaction',
    'English':   'EU-heavy 2022 reaction',
    'Danish':    'EU-heavy 2022 reaction',
    'Czech':     'EU-heavy 2022 reaction',
    'Swedish':   'EU-heavy 2022 reaction',
    'Polish':    'EU-heavy 2022 reaction',

    # Cluster 5: geostrategic neighbours with sustained attention
    'Romanian': 'Geostrategic sustained attention',
    'Greek':    'Geostrategic sustained attention'
}

print('Cluster mapping loaded.')

In [ ]:
# --- Load and aggregate data ---
df_all = pd.read_csv(CHOSEN_WORDS_DAILY_FILE)
df_all['date'] = pd.to_datetime(df_all['date'])

iso_to_name = (
    pd.read_csv(WORD_FORMS_ALL, usecols=['ISO', 'Language'])
    .drop_duplicates()
    .set_index('ISO')['Language']
)

df_all['language'] = df_all['language_ISO'].map(iso_to_name)
df_all = df_all.dropna(subset=['language']).copy()

DF = (
    df_all.groupby(['date', 'language'], as_index=False)[['freq', 'count', 'count_no_rt', 'freq_no_rt']]
    .sum()
)

print(DF.head(3))

In [ ]:
# --- Build 2014 and 2022 windows; apply required 2014 shift +8y,+6d ---
DF2014 = DF[(DF['date'] >= CENTRAL_DATES['2014'] - pd.Timedelta(weeks=4)) &
            (DF['date'] <= CENTRAL_DATES['2014'] + pd.Timedelta(weeks=4))].copy()

DF2022 = DF[(DF['date'] >= CENTRAL_DATES['2022'] - pd.Timedelta(weeks=4)) &
            (DF['date'] <= CENTRAL_DATES['2022'] + pd.Timedelta(weeks=4))].copy()

DF2014['date'] = DF2014['date'] + pd.DateOffset(years=8, days=6)

# Keep old normalization behavior available (freq/local_mean), though plotting uses raw freq
def add_local_norm(df, freq_col='freq', lang_col='language'):
    out = df.copy()
    local_mean = out.groupby(lang_col)[freq_col].transform('mean')
    out['normalized_loc_freq'] = out[freq_col] / local_mean
    out['normalized_overal_freq_log'] = np.log10(out['normalized_loc_freq'])
    out['freq_log'] = np.log10(out[freq_col].replace(0, np.nan))
    return out

DF2014 = add_local_norm(DF2014)
DF2022 = add_local_norm(DF2022)

print('Windows ready:', DF2014['date'].min().date(), DF2014['date'].max().date(), '|', DF2022['date'].min().date(), DF2022['date'].max().date())

In [ ]:
# --- Combine and tag clusters ---
DF2014['cluster'] = DF2014['language'].map(lang2cluster)
DF2022['cluster'] = DF2022['language'].map(lang2cluster)
DF2014['year'] = '2014'
DF2022['year'] = '2022'

df = pd.concat([DF2014, DF2022], ignore_index=True)
df = df.dropna(subset=['cluster']).copy()
df['freq'] = df['freq'].replace(0, np.nan)

cluster_ts = (
    df.groupby(['cluster', 'year', 'date'])['freq']
    .agg(
        mean='mean',
        q10=lambda x: x.quantile(0.10),
        q90=lambda x: x.quantile(0.90)
    )
    .reset_index()
)

clusters = cluster_ts['cluster'].dropna().unique()
print('Clusters:', list(clusters))

In [ ]:
# --- Plot Figure S6 with numeric-millisecond x-axis (Kaleido-safe) ---
fig = make_subplots(
    rows=len(clusters), cols=1,
    shared_xaxes=True,
    vertical_spacing=0.02
)

mean_vals = cluster_ts['mean']
y_min = mean_vals[mean_vals > 0].min()
y_max = mean_vals.max()
log_y_min = np.log10(y_min) - 0.95
log_y_max = np.log10(y_max) + 0.3

feb_24_2022 = pd.to_datetime('2022-02-24')
date_intervals = (
    [feb_24_2022 - pd.Timedelta(weeks=i) for i in range(4, 0, -1)]
    + [feb_24_2022]
    + [feb_24_2022 + pd.Timedelta(weeks=i) for i in range(1, 5)]
)
date_labels = ['-4w', '-3w', '-2w', '-1w', 'Invasion', '+1w', '+2w', '+3w', '+4w']
date_interval_nums = [d.value // 10**6 for d in date_intervals]

for i, cl in enumerate(clusters, start=1):
    for year, color in [('2014', '#4d99c6'), ('2022', '#bd2f36')]:
        langs = df.loc[(df['cluster'] == cl) & (df['year'] == year), 'language'].unique()

        for lang in langs:
            dlang = df.query('cluster == @cl and year == @year and language == @lang')
            x_numeric = dlang['date'].astype('int64') // 10**6

            fig.add_trace(
                go.Scatter(
                    x=x_numeric,
                    y=dlang['freq'],
                    mode='lines',
                    line=dict(color=color, width=2),
                    opacity=0.2,
                    connectgaps=False,
                    showlegend=False
                ),
                row=i, col=1
            )

        dmean = cluster_ts.query('cluster == @cl and year == @year')
        x_mean_numeric = dmean['date'].astype('int64') // 10**6

        fig.add_trace(
            go.Scatter(
                x=x_mean_numeric,
                y=dmean['mean'],
                mode='lines',
                line=dict(color=color, width=3),
                connectgaps=False,
                showlegend=False
            ),
            row=i, col=1
        )

    fig.update_xaxes(
        row=i, col=1,
        tickmode='array',
        tickvals=date_interval_nums,
        ticktext=date_labels,
        showgrid=True,
        gridcolor='lightgrey',
        gridwidth=1,
        tickfont=dict(size=26),
        title_font=dict(size=28),
        type='linear'
    )

    fig.update_yaxes(
        row=i, col=1,
        title_text=cl,
        type='log',
        range=[log_y_min, log_y_max],
        tickfont=dict(size=26),
        title_font=dict(size=36)
    )

fig.update_layout(
    width=800,
    height=300 * len(clusters),
    showlegend=False,
    margin=dict(l=150, r=30, t=30, b=70),
    title='',
    font=dict(size=26, family='Arial')
)

fig.update_xaxes(fixedrange=True)
fig.update_yaxes(fixedrange=True)
fig.full_figure_for_development(warn=False)

for fmt in ['pdf', 'png', 'svg']:
    out_dir = S6_BASE / fmt
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / f'trendclusters.{fmt}'
    fig.write_image(
        out_file,
        format=fmt,
        engine='kaleido',
        width=1600,
        height=600 * len(clusters),
        scale=2
    )
    print(f'Saved {fmt.upper()} to: {out_file}')

fig.show()